# 05 – Construcción de features ricas para riesgo de hipertensión (NHANES)

## Objetivo general

El objetivo de este cuaderno es construir un conjunto de *features ricas* para predecir riesgo de hipertensión arterial a partir de datos de NHANES (p. ej. ciclos 2015–2016 y 2017–2018).  

La idea es aproximarse a un **“cuestionario de riesgo de hipertensión”** que pueda responder una persona sin acudir previamente al médico ni medirse la presión arterial.

Por este motivo:

1. Solo se utilizarán variables que la persona pueda declarar por sí misma:
   - Hábitos (tabaco, alcohol, sueño).
   - Síntomas y antecedentes personales.
   - Comorbilidades ya conocidas.
   - Variables sociodemográficas básicas.

2. **Se evitarán explícitamente variables que sean casi equivalentes a la etiqueta `HTN_label`**, como:
   - “Alguna vez le han dicho que tiene presión alta / hipertensión”.
   - “Actualmente toma medicamentos para la presión”.
   Estas variables contienen información casi idéntica al *label* y producirían **data leakage**, inflando artificialmente el desempeño del modelo.

El resultado del cuaderno será un archivo:

- `data/05_model_input/nhanes_features_rich.csv`

listo para ser usado en modelos de regresión y clasificación de riesgo de hipertensión, e integrable posteriormente en una *pipeline* de Kedro.


In [34]:
# 1. Importaciones básicas y configuración de entorno

import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

plt.style.use("default")

# Asegurar que estamos en la raíz del proyecto (no en /notebooks)
cwd = os.getcwd()
if cwd.endswith("notebooks"):
    os.chdir("..")
    print("Cambiando directorio de trabajo a la raíz del proyecto.")
print("Working dir:", os.getcwd())


Working dir: c:\Users\Ella es ella\Documents\kedro_projects\nhanes-htn


In [37]:
# 2. Helpers generales para carga y overview de datasets

def load_nhanes_module(stem_base, cycles=("I", "J"), root="data/01_raw/richer_nhanes"):
    """
    Carga uno o varios módulos NHANES (p. ej. SMQ_I, SMQ_J) desde data/01_raw.
    - Intenta primero .csv y luego .XPT/.xpt.
    - Concatena verticalmente todos los ciclos encontrados.

    Parámetros:
        stem_base: prefijo del archivo sin sufijo de ciclo ni extensión (p. ej. "SMQ").
        cycles: tupla con códigos de ciclo ("I"=2015–2016, "J"=2017–2018, etc.).
        root: carpeta base donde están los archivos.

    Retorna:
        DataFrame concatenado con todos los ciclos encontrados.
    """
    frames = []
    for c in cycles:
        base = f"{stem_base}_{c}"
        candidates = [
            os.path.join(root, base + ".csv"),
            os.path.join(root, base + ".CSV"),
        ]
        for path in candidates:
            if os.path.exists(path):
                print(f"Cargando {path!r}")
                if path.lower().endswith(".csv"):
                    df = pd.read_csv(path)
                else:
                    # Archivos XPT de NHANES
                    df = pd.read_sas(path, format="xport", encoding="utf-8")
                df["cycle_code"] = c
                frames.append(df)
                break
    if not frames:
        raise FileNotFoundError(
            f"No se encontró ningún archivo para {stem_base}_{{{','.join(cycles)}}} "
            f"en la carpeta {root}."
        )
    df_all = pd.concat(frames, ignore_index=True)
    return df_all


def quick_overview(df, name, n_head=5):
    """Imprime shape, info y head() de un DataFrame."""
    print(f"\n=== Overview de {name} ===")
    print("Shape:", df.shape)
    display(df.head(n_head))
    print("\nInfo():")
    print(df.info())


In [38]:
# 3. Carga del dataset demográfico/antropométrico base
#    (usamos el dataset ya integrado de 03_primary para mantener coherencia con HTN_label)

core_path = "data/03_primary/nhanes_supervised_dataset.csv"
core = pd.read_csv(core_path)

quick_overview(core, "core (nhanes_supervised_dataset)")


=== Overview de core (nhanes_supervised_dataset) ===
Shape: (11268, 34)


,id,gender_code,age_years,race_ethnicity_code,education_level_code,income_poverty_ratio,weight_kg,height_cm,bmi,waist_cm,...,vigorous_recreation_days,vigorous_recreation_minutes,moderate_recreation_code,moderate_recreation_days,moderate_recreation_minutes,sedentary_minutes,cycle,SBP_mean,DBP_mean,HTN_label
0,83732,1,62,3,5,4.39,94.8,184.5,27.8,101.1,...,0.0,0.0,1,6.0,30.0,480.0,2015_2016,122.666667,65.333333,0
1,83733,1,53,3,3,1.32,90.4,171.4,30.8,107.9,...,0.0,0.0,2,0.0,0.0,300.0,2015_2016,140.000000,86.000000,1
2,83734,1,78,3,3,1.51,83.4,170.1,28.8,116.5,...,0.0,0.0,2,0.0,0.0,480.0,2015_2016,135.333333,45.333333,0
3,83735,2,56,3,5,5.00,109.8,160.9,42.4,110.1,...,0.0,0.0,2,0.0,0.0,480.0,2015_2016,134.000000,70.000000,0
4,83736,2,42,4,4,1.23,55.2,164.9,20.3,80.4,...,0.0,0.0,2,0.0,0.0,540.0,2015_2016,104.000000,60.000000,0



Info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11268 entries, 0 to 11267
Data columns (total 34 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id                           11268 non-null  int64  
 1   gender_code                  11268 non-null  int64  
 2   age_years                    11268 non-null  int64  
 3   race_ethnicity_code          11268 non-null  int64  
 4   education_level_code         11268 non-null  int64  
 5   income_poverty_ratio         11268 non-null  float64
 6   weight_kg                    11268 non-null  float64
 7   height_cm                    11268 non-null  float64
 8   bmi                          11268 non-null  float64
 9   waist_cm                     11268 non-null  float64
 10  BPXSY1                       10300 non-null  float64
 11  BPXDI1                       10300 non-null  float64
 12  BPXSY2                       10663 non-null  float64
 13  BPXDI2 

## 2. Carga y overview de módulos de cuestionario

En las siguientes celdas se cargan los módulos NHANES adicionales que aportan información tipo “cuestionario”:

- **SMQ_I / SMQ_J** y **SMQFAM_I / SMQFAM_J**: tabaco.
- **ALQ_I / ALQ_J**: consumo de alcohol.
- **SLQ_I / SLQ_J**: sueño.
- **DPQ_I / DPQ_J**: depresión (PHQ-9).
- **MCQ_I / MCQ_J**: condiciones médicas generales.

Para cada módulo se mostrará:

1. Dimensiones (`shape`).
2. Primeras filas (`head()`).
3. Esquema de columnas (`info()`).

Esto permite verificar que las variables necesarias estén presentes y detectar posibles diferencias de nombres entre ciclos.


In [39]:
# 3.1 Carga de módulos de cuestionario

smq = load_nhanes_module("SMQ")        # Smoking Questionnaire
smqfam = load_nhanes_module("SMQFAM")  # Exposición a humo de tabaco
alq = load_nhanes_module("ALQ")        # Alcohol Use
slq = load_nhanes_module("SLQ")        # Sleep
dpq = load_nhanes_module("DPQ")        # Depression Screener (PHQ-9)

# Overviews
quick_overview(smq, "SMQ (tabaco)")
quick_overview(smqfam, "SMQFAM (humo ambiental)")
quick_overview(alq, "ALQ (alcohol)")
quick_overview(slq, "SLQ (sueño)")
quick_overview(dpq, "DPQ (PHQ-9)")


Cargando 'data/01_raw/richer_nhanes\\SMQ_I.csv'
Cargando 'data/01_raw/richer_nhanes\\SMQ_J.csv'
Cargando 'data/01_raw/richer_nhanes\\SMQFAM_I.csv'
Cargando 'data/01_raw/richer_nhanes\\SMQFAM_J.csv'
Cargando 'data/01_raw/richer_nhanes\\ALQ_I.csv'
Cargando 'data/01_raw/richer_nhanes\\ALQ_J.csv'
Cargando 'data/01_raw/richer_nhanes\\SLQ_I.csv'
Cargando 'data/01_raw/richer_nhanes\\SLQ_J.csv'
Cargando 'data/01_raw/richer_nhanes\\DPQ_I.csv'
Cargando 'data/01_raw/richer_nhanes\\DPQ_J.csv'

=== Overview de SMQ (tabaco) ===
Shape: (13725, 43)


,SEQN,SMQ020,SMD030,SMQ040,SMQ050Q,SMQ050U,SMD055,SMD057,SMQ078,SMD641,...,SMQ935,SMQ080,SMQ890,SMQ895,SMQ900,SMQ905,SMQ910,SMQ915,SMAQUEX2,cycle_code
0,83732.0,1.0,22.0,3.0,22.0,4.0,40.0,20.0,NaN,NaN,...,NaN,NaN,2.0,NaN,2.0,NaN,1.0,5.397605e-79,1.0,I
1,83733.0,1.0,20.0,1.0,NaN,NaN,NaN,NaN,2.0,30.0,...,NaN,NaN,1.0,3.000000e+01,2.0,NaN,2.0,NaN,1.0,I
2,83734.0,1.0,14.0,3.0,16.0,4.0,61.0,30.0,NaN,NaN,...,NaN,NaN,1.0,5.397605e-79,2.0,NaN,1.0,5.397605e-79,1.0,I
3,83735.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,1.0,I
4,83736.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,7.000000e+00,2.0,NaN,2.0,NaN,1.0,I



Info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13725 entries, 0 to 13724
Data columns (total 43 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   SEQN        13725 non-null  float64
 1   SMQ020      11848 non-null  float64
 2   SMD030      4781 non-null   float64
 3   SMQ040      4781 non-null   float64
 4   SMQ050Q     2660 non-null   float64
 5   SMQ050U     2517 non-null   float64
 6   SMD055      1201 non-null   float64
 7   SMD057      2660 non-null   float64
 8   SMQ078      1621 non-null   float64
 9   SMD641      2225 non-null   float64
 10  SMD650      2131 non-null   float64
 11  SMD093      2121 non-null   float64
 12  SMDUPCA     13725 non-null  object 
 13  SMD100BR    13725 non-null  object 
 14  SMD100FL    1933 non-null   float64
 15  SMD100MN    1933 non-null   float64
 16  SMD100LN    1933 non-null   float64
 17  SMD100TR    1447 non-null   float64
 18  SMD100NI    1447 non-null   float64
 19  SMD100CO    1447

,SEQN,SMD460,SMD470,SMD480,cycle_code
0,83732.0,5.397605e-79,NaN,NaN,I
1,83733.0,1.000000e+00,1.0,7.0,I
2,83734.0,1.000000e+00,1.0,7.0,I
3,83735.0,5.397605e-79,NaN,NaN,I
4,83736.0,3.000000e+00,3.0,3.0,I



Info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19225 entries, 0 to 19224
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   SEQN        19225 non-null  float64
 1   SMD460      18440 non-null  float64
 2   SMD470      5400 non-null   float64
 3   SMD480      1914 non-null   float64
 4   cycle_code  19225 non-null  object 
dtypes: float64(4), object(1)
memory usage: 751.1+ KB
None

=== Overview de ALQ (alcohol) ===
Shape: (11268, 18)


...


Info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11268 entries, 0 to 11267
Data columns (total 18 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   SEQN        11268 non-null  float64
 1   ALQ101      5208 non-null   float64
 2   ALQ110      1731 non-null   float64
 3   ALQ120Q     4224 non-null   float64
 4   ALQ120U     3376 non-null   float64
 5   ALQ130      6874 non-null   float64
 6   ALQ141Q     3377 non-null   float64
 7   ALQ141U     1340 non-null   float64
 8   ALQ151      8766 non-null   float64
 9   ALQ160      1352 non-null   float64
 10  cycle_code  11268 non-null  object 
 11  ALQ111      5130 non-null   float64
 12  ALQ121      4545 non-null   float64
 13  ALQ142      3495 non-null   float64
 14  ALQ270      1439 non-null   float64
 15  ALQ280      1439 non-null   float64
 16  ALQ290      522 non-null    float64
 17  ALQ170      3487 non-null   float64
dtypes: float64(17), object(1)
memory usage: 1.5+ MB
None

===

,SEQN,SLQ300,SLQ310,SLD012,SLQ030,SLQ040,SLQ050,SLQ120,cycle_code,SLQ320,SLQ330,SLD013
0,83732.0,b'23:30',b'05:00',5.5,2.000000e+00,9.000000e+00,1.0,3.000000e+00,I,NaN,NaN,NaN
1,83733.0,b'23:00',b'07:00',8.0,1.000000e+00,5.397605e-79,2.0,5.397605e-79,I,NaN,NaN,NaN
2,83734.0,b'22:30',b'05:30',7.0,5.397605e-79,9.000000e+00,2.0,3.000000e+00,I,NaN,NaN,NaN
3,83735.0,b'23:30',b'06:00',6.5,9.000000e+00,1.000000e+00,1.0,4.000000e+00,I,NaN,NaN,NaN
4,83736.0,b'99999',b'06:00',NaN,9.000000e+00,9.000000e+00,1.0,1.000000e+00,I,NaN,NaN,NaN



Info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12488 entries, 0 to 12487
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   SEQN        12488 non-null  float64
 1   SLQ300      12488 non-null  object 
 2   SLQ310      12488 non-null  object 
 3   SLD012      12407 non-null  float64
 4   SLQ030      12488 non-null  float64
 5   SLQ040      12488 non-null  float64
 6   SLQ050      12488 non-null  float64
 7   SLQ120      12488 non-null  float64
 8   cycle_code  12488 non-null  object 
 9   SLQ320      6161 non-null   object 
 10  SLQ330      6161 non-null   object 
 11  SLD013      6104 non-null   float64
dtypes: float64(7), object(5)
memory usage: 1.1+ MB
None

=== Overview de DPQ (PHQ-9) ===
Shape: (11268, 12)


,SEQN,DPQ010,DPQ020,DPQ030,DPQ040,DPQ050,DPQ060,DPQ070,DPQ080,DPQ090,DPQ100,cycle_code
0,83732.0,5.397605e-79,5.397605e-79,5.397605e-79,1.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,I
1,83733.0,1.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79,1.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,I
2,83734.0,5.397605e-79,5.397605e-79,5.397605e-79,1.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,1.000000e+00,I
3,83735.0,1.000000e+00,1.000000e+00,2.000000e+00,2.000000e+00,1.000000e+00,3.000000e+00,2.000000e+00,5.397605e-79,1.000000e+00,5.397605e-79,I
4,83736.0,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,3.000000e+00,5.397605e-79,1.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79,I



Info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11268 entries, 0 to 11267
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   SEQN        11268 non-null  float64
 1   DPQ010      10258 non-null  float64
 2   DPQ020      10257 non-null  float64
 3   DPQ030      10257 non-null  float64
 4   DPQ040      10254 non-null  float64
 5   DPQ050      10254 non-null  float64
 6   DPQ060      10253 non-null  float64
 7   DPQ070      10252 non-null  float64
 8   DPQ080      10252 non-null  float64
 9   DPQ090      10251 non-null  float64
 10  DPQ100      6941 non-null   float64
 11  cycle_code  11268 non-null  object 
dtypes: float64(11), object(1)
memory usage: 1.0+ MB
None


## 3. Selección de variables por módulo y definición de diccionarios de columnas

A continuación se definen diccionarios con los **nombres estándar de columnas NHANES** que se usarán para derivar las features.  

Si en tus archivos los nombres son diferentes, solo debes modificar estos diccionarios; el resto del código seguirá funcionando.

Las listas están pensadas para NHANES 2015–2016 / 2017–2018, pero la estructura es suficientemente flexible para adaptar otros ciclos.


In [40]:
# 3. Diccionarios de nombres de columnas (ajustar si tus datasets usan otros nombres)

# 3.1 Tabaco (SMQ / SMQFAM)
SMOKING_COLS = {
    "id": "SEQN",
    "ever_100_cigs": "SMQ020",         # 1 = Yes, 2 = No, 7/9 = missing
    "current_status": "SMQ040",        # 1 = Every day, 2 = Some days, 3 = Not at all
    "age_start_regular": "SMD030",     # edad de inicio consumo regular (si existe)
    "age_quit": "SMD055",              # edad de cese (si existe)
    "cigs_per_day": "SMD650",          # cigarrillos/día (ajusta si usas otra variable)
}

# Tabaco pasivo en el hogar (SMQFAM, SMD***)
SMOKING_FAM_COLS = {
    "id": "SEQN",
    "n_smokers_home": "SMD460",        # # personas que fuman tabaco en el hogar
    "n_smokers_inside": "SMD470",      # # personas que fuman dentro de la casa
    "days_smoked_inside": "SMD480",    # # días en la última semana que alguien fumó dentro
}

# 3.2 Alcohol (ALQ)
ALCOHOL_COLS = {
    "id": "SEQN",
    "freq_12m": "ALQ120Q",             # frecuencia (número)
    "freq_12m_unit": "ALQ120U",        # unidad (día/semana/mes/año)
    "drinks_per_day": "ALQ130",        # tragos típicos por día de consumo
    "binge_days": "ALQ141Q",           # # de días con ≥4/5 tragos
    "binge_unit": "ALQ141U",           # unidad (semana/mes/año)
    # ALQ160 podrías usarlo como validación adicional; aquí no es imprescindible
}

# 3.3 Sueño (SLQ/SLD)
SLEEP_COLS = {
    "id": "SEQN",
    "hours_sleep": "SLD012",           # horas habituales de sueño
    "snoring_freq": "SLQ030",          # frecuencia de ronquidos
    "apnea_obs": "SLQ040",            # frecuencia de "snort or stop breathing"
    "daytime_sleepiness": "SLQ120",    # somnolencia diurna
}

# 3.4 Depresión (DPQ – PHQ-9)
DPQ_ITEMS = {
    "id": "SEQN",
    "items": [
        "DPQ010", "DPQ020", "DPQ030",
        "DPQ040", "DPQ050", "DPQ060",
        "DPQ070", "DPQ080", "DPQ090",
    ],
}



In [41]:
# 4. Función general para mapear códigos NHANES de "no sabe / no responde / rehúsa" a NaN

def nhanes_missing_to_nan(series, extra_missing=None):
    """
    Reemplaza códigos NHANES típicos de 'missing' por NaN.
    extra_missing permite agregar otros códigos específicos del ítem.
    """
    missing_codes = {7, 9, 77, 79, 97, 99, 777, 999, 9999}
    if extra_missing:
        missing_codes.update(extra_missing)
    return series.replace(list(missing_codes), np.nan)


## 3.1 Tabaco – SMQ / SMQFAM

Objetivos:

1. Derivar, a partir de SMQ:
   - Haber fumado ≥100 cigarrillos en la vida (ever smoker).
   - Situación actual: fuma a diario / algunos días / no fuma.
   - Edad de inicio del consumo regular.
   - Edad de cese o años desde que dejó de fumar.
   - Cigarrillos por día (cuando fumaba más o en los últimos 30 días).

2. A partir de SMQFAM:
   - Exposición a humo de tabaco en el hogar.
   - Exposición a humo de tabaco en el trabajo.

Features derivadas:

- `current_smoker` (0/1).
- `former_smoker` (0/1).
- `never_smoker` (0/1).
- `pack_years_aprox` (proxy simple: cigarrillos/día × años fumando / 20).
- `secondhand_smoke_home` (0/1).
- `secondhand_smoke_work` (0/1).

Estas variables describen **conductas y exposiciones**, no mediciones de presión arterial, por lo que **no introducen data leakage** respecto de `HTN_label`.


In [42]:
# 3.1 Derivación de features de tabaco (SMQ + SMQFAM)

def build_smoking_features(smq_df, smqfam_df, core_df):
    """
    Construye features de tabaco a partir de SMQ, SMQFAM y datos demográficos (para la edad).
    Retorna un DataFrame con una fila por participante (id/SEQN) y columnas derivadas.
    """

    # ---------- Subset y renombrado SMQ ----------
    cols = [SMOKING_COLS["id"]]
    for key in ["ever_100_cigs", "current_status", "age_start_regular", "age_quit", "cigs_per_day"]:
        col = SMOKING_COLS.get(key)
        if col in smq_df.columns:
            cols.append(col)

    smq_sub = smq_df[cols].copy()
    smq_sub = smq_sub.rename(columns={SMOKING_COLS["id"]: "id"})

    # Limpiar códigos de missing
    for key in ["ever_100_cigs", "current_status", "age_start_regular", "age_quit", "cigs_per_day"]:
        col = SMOKING_COLS.get(key)
        if col in smq_sub.columns:
            smq_sub[col] = nhanes_missing_to_nan(smq_sub[col])

    # Merge con edad actual desde el core
    age_map = core_df[["id", "age_years"]].drop_duplicates()
    smq_sub = smq_sub.merge(age_map, on="id", how="left")

    ever_col = SMOKING_COLS.get("ever_100_cigs")
    status_col = SMOKING_COLS.get("current_status")

    # ---------- Indicadores ever/current/former/never ----------
    if ever_col in smq_sub.columns:
        smq_sub["ever_smoker"] = np.where(smq_sub[ever_col] == 1, 1, 0)
    else:
        smq_sub["ever_smoker"] = np.nan

    smq_sub["current_smoker"] = np.nan
    smq_sub["former_smoker"] = np.nan
    smq_sub["never_smoker"] = np.nan

    if status_col in smq_sub.columns:
        # 1 = every day, 2 = some days, 3 = not at all
        smq_sub["current_smoker"] = np.where(smq_sub[status_col].isin([1, 2]), 1,
                                             np.where(smq_sub[status_col].isin([3]), 0, np.nan))

        smq_sub["former_smoker"] = np.where(
            (smq_sub["ever_smoker"] == 1) & (smq_sub["current_smoker"] == 0),
            1,
            np.where(smq_sub["ever_smoker"].isna() | smq_sub["current_smoker"].isna(), np.nan, 0),
        )

        smq_sub["never_smoker"] = np.where(
            (smq_sub["ever_smoker"] == 0) & (~smq_sub[status_col].isna()),
            1,
            np.where(smq_sub["ever_smoker"].isna() | smq_sub[status_col].isna(), np.nan, 0),
        )

    # ---------- Años fumando y pack-years aprox ----------
    age_start_col = SMOKING_COLS.get("age_start_regular")
    age_quit_col = SMOKING_COLS.get("age_quit")
    cigs_col = SMOKING_COLS.get("cigs_per_day")

    smq_sub["years_smoking_aprox"] = np.nan

    if age_start_col in smq_sub.columns:
        # Fumador actual: edad actual - edad inicio
        mask_current = smq_sub["current_smoker"] == 1
        smq_sub.loc[mask_current, "years_smoking_aprox"] = (
            smq_sub.loc[mask_current, "age_years"] - smq_sub.loc[mask_current, age_start_col]
        )

        # Exfumador: edad cese - edad inicio
        if age_quit_col in smq_sub.columns:
            mask_former = (smq_sub["former_smoker"] == 1) & (~smq_sub[age_quit_col].isna())
            smq_sub.loc[mask_former, "years_smoking_aprox"] = (
                smq_sub.loc[mask_former, age_quit_col] - smq_sub.loc[mask_former, age_start_col]
            )

    smq_sub.loc[smq_sub["years_smoking_aprox"] < 0, "years_smoking_aprox"] = np.nan

    if cigs_col in smq_sub.columns:
        smq_sub["pack_years_aprox"] = (
            smq_sub[cigs_col] * smq_sub["years_smoking_aprox"] / 20.0
        )
    else:
        smq_sub["pack_years_aprox"] = np.nan

    # pack_years_aprox = 0 para nunca fumadores
    smq_sub.loc[smq_sub["never_smoker"] == 1, "pack_years_aprox"] = 0.0

    # ---------- Tabaco pasivo (SMQFAM) ----------
    fam_cols = [SMOKING_FAM_COLS["id"]]
    for key in ["n_smokers_home", "n_smokers_inside", "days_smoked_inside"]:
        col = SMOKING_FAM_COLS.get(key)
        if col in smqfam_df.columns:
            fam_cols.append(col)

    smqfam_sub = smqfam_df[fam_cols].copy().rename(
        columns={SMOKING_FAM_COLS["id"]: "id"}
    )

    # Missing → NaN
    for key in ["n_smokers_home", "n_smokers_inside", "days_smoked_inside"]:
        col = SMOKING_FAM_COLS.get(key)
        if col in smqfam_sub.columns:
            smqfam_sub[col] = nhanes_missing_to_nan(smqfam_sub[col])

    # 0 = nadie; >=1 = exposición; NaN = desconocido
    if SMOKING_FAM_COLS.get("n_smokers_home") in smqfam_sub.columns:
        col = SMOKING_FAM_COLS["n_smokers_home"]
        smqfam_sub["secondhand_smoke_home"] = np.where(
            smqfam_sub[col].isna(), np.nan,
            np.where(smqfam_sub[col] >= 1, 1, 0)
        )
    else:
        smqfam_sub["secondhand_smoke_home"] = np.nan

    if SMOKING_FAM_COLS.get("n_smokers_inside") in smqfam_sub.columns:
        col = SMOKING_FAM_COLS["n_smokers_inside"]
        smqfam_sub["secondhand_smoke_inside"] = np.where(
            smqfam_sub[col].isna(), np.nan,
            np.where(smqfam_sub[col] >= 1, 1, 0)
        )
    else:
        smqfam_sub["secondhand_smoke_inside"] = np.nan

    # ---------- Frecuencia de humo dentro de casa ----------
    inside_col = SMOKING_FAM_COLS.get("n_smokers_inside")
    days_col = SMOKING_FAM_COLS.get("days_smoked_inside")

    smqfam_sub["smoke_inside_freq_cat"] = np.nan

    if inside_col in smqfam_sub.columns and days_col in smqfam_sub.columns:
        inside = smqfam_sub[inside_col]
        days = smqfam_sub[days_col]

        # Nadie fuma dentro de casa → 0_days (aunque SMD480 sea NaN por skip)
        mask_no_inside = inside == 0
        smqfam_sub.loc[mask_no_inside, "smoke_inside_freq_cat"] = "0_days"

        # Hay fumadores dentro de casa
        mask_has_inside = inside >= 1

        smqfam_sub.loc[mask_has_inside & (days == 0), "smoke_inside_freq_cat"] = "0_days"
        smqfam_sub.loc[mask_has_inside & days.between(1, 3, inclusive="both"), "smoke_inside_freq_cat"] = "1-3_days"
        smqfam_sub.loc[mask_has_inside & days.between(4, 7, inclusive="both"), "smoke_inside_freq_cat"] = "4-7_days"
        # Si days es NaN en alguien con inside>=1, se deja NaN (desconocido)

    # ---------- Merge final ----------
    smoking_features = (
        smq_sub[["id", "current_smoker", "former_smoker", "never_smoker", "pack_years_aprox"]]
        .merge(
            smqfam_sub[["id", "secondhand_smoke_home", "secondhand_smoke_inside", "smoke_inside_freq_cat"]],
            on="id",
            how="left",
        )
        .drop_duplicates(subset="id")
    )

    return smoking_features


smoking_features = build_smoking_features(smq, smqfam, core)
quick_overview(smoking_features, "Features de tabaco")



=== Overview de Features de tabaco ===
Shape: (13725, 8)


C:\Users\Ella es ella\AppData\Local\Temp\ipykernel_11556\3910952652.py:140: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0_days' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  smqfam_sub.loc[mask_no_inside, "smoke_inside_freq_cat"] = "0_days"


,id,current_smoker,former_smoker,never_smoker,pack_years_aprox,secondhand_smoke_home,secondhand_smoke_inside,smoke_inside_freq_cat
0,83732.0,0.0,1.0,0.0,NaN,0.0,NaN,NaN
1,83733.0,1.0,0.0,0.0,33.0,1.0,1.0,NaN
2,83734.0,0.0,1.0,0.0,NaN,1.0,1.0,NaN
3,83735.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN
4,83736.0,NaN,NaN,NaN,NaN,1.0,1.0,1-3_days



Info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13725 entries, 0 to 13724
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       13725 non-null  float64
 1   current_smoker           4781 non-null   float64
 2   former_smoker            4781 non-null   float64
 3   never_smoker             4781 non-null   float64
 4   pack_years_aprox         1879 non-null   float64
 5   secondhand_smoke_home    13128 non-null  float64
 6   secondhand_smoke_inside  3899 non-null   float64
 7   smoke_inside_freq_cat    451 non-null    object 
dtypes: float64(7), object(1)
memory usage: 857.9+ KB
None


## 3.2 Alcohol – ALQ

Objetivos:

1. Seleccionar variables de los últimos 12 meses:
   - Frecuencia de consumo (veces/mes o días/semana).
   - Tragos típicos por día de consumo.
   - Máximo de tragos en un mismo día.
   - Número de días con consumo elevado (si existe).

2. Derivar:
   - `alcohol_level` (categórico: none / low / moderate / heavy).
   - `binge_drinker` (0/1).

Criterio propuesto (simplificado y orientado a cuestionario):

- Se estima una **tasa aproximada de tragos por semana**.
- Cortes sugeridos:
  - 0 tragos/semana → `none`.
  - >0 y ≤7 tragos/semana → `low`.
  - >7 y ≤14 tragos/semana → `moderate`.
  - >14 tragos/semana → `heavy`.

`binge_drinker` se define si el máximo número de tragos en un día (`max_drinks_day`) es ≥5.

Estas variables describen patrones de consumo y no incluyen mediciones de presión arterial, por lo que **no producen data leakage**.


In [43]:
# 3.2 Derivación de features de alcohol (ALQ)

def build_alcohol_features(alq_df):
    cols = [ALCOHOL_COLS["id"]]
    for key in ["freq_12m", "freq_12m_unit", "drinks_per_day", "binge_days", "binge_unit"]:
        col = ALCOHOL_COLS.get(key)
        if col in alq_df.columns:
            cols.append(col)

    alq_sub = alq_df[cols].copy().rename(columns={ALCOHOL_COLS["id"]: "id"})

    # Limpiar missing
    for key in ["freq_12m", "freq_12m_unit", "drinks_per_day", "binge_days", "binge_unit"]:
        col = ALCOHOL_COLS.get(key)
        if col in alq_sub.columns:
            alq_sub[col] = nhanes_missing_to_nan(alq_sub[col])

    freq_col = ALCOHOL_COLS.get("freq_12m")
    unit_col = ALCOHOL_COLS.get("freq_12m_unit")

    # ---------- Días de consumo por semana ----------
    alq_sub["drinking_days_per_week"] = np.nan

    if freq_col in alq_sub.columns and unit_col in alq_sub.columns:
        f = alq_sub[freq_col]
        u = alq_sub[unit_col]

        # Supuesto de unidad (ajusta según codebook exacto):
        # 1 = per day, 2 = per week, 3 = per month, 4 = per year
        days_per_week = np.full(len(alq_sub), np.nan, dtype=float)
        days_per_week = np.where(u == 1, f * 7, days_per_week)
        days_per_week = np.where(u == 2, f, days_per_week)
        days_per_week = np.where(u == 3, f / 4.0, days_per_week)
        days_per_week = np.where(u == 4, f / 52.0, days_per_week)

        alq_sub["drinking_days_per_week"] = days_per_week

    # ---------- Tragos por semana ----------
    drinks_day_col = ALCOHOL_COLS.get("drinks_per_day")
    if drinks_day_col in alq_sub.columns:
        alq_sub["drinks_per_drinking_day"] = alq_sub[drinks_day_col]
    else:
        alq_sub["drinks_per_drinking_day"] = np.nan

    alq_sub["drinks_per_week"] = (
        alq_sub["drinking_days_per_week"] * alq_sub["drinks_per_drinking_day"]
    )

    # ---------- Binge drinking con ALQ141Q/ALQ141U ----------
    binge_days_col = ALCOHOL_COLS.get("binge_days")
    binge_unit_col = ALCOHOL_COLS.get("binge_unit")

    alq_sub["binge_days_per_month"] = np.nan
    alq_sub["binge_drinker"] = np.nan

    if binge_days_col in alq_sub.columns and binge_unit_col in alq_sub.columns:
        b = alq_sub[binge_days_col]
        u = alq_sub[binge_unit_col]

        # Misma lógica de unidades
        binge_per_year = np.full(len(alq_sub), np.nan, dtype=float)
        binge_per_year = np.where(u == 1, b * 365, binge_per_year)   # diario
        binge_per_year = np.where(u == 2, b * 52, binge_per_year)    # semanal
        binge_per_year = np.where(u == 3, b * 12, binge_per_year)    # mensual
        binge_per_year = np.where(u == 4, b, binge_per_year)         # anual

        binge_per_month = binge_per_year / 12.0
        alq_sub["binge_days_per_month"] = binge_per_month

        # Binge_drinker = 1 si ≥1 día de binge al mes; 0 si <1; NaN si no sabemos
        alq_sub["binge_drinker"] = np.where(
            binge_per_month >= 1, 1,
            np.where(binge_per_month < 1, 0, np.nan)
        )

    # ---------- Categoría de consumo total ----------
    def categorize_alcohol(row):
        dpw = row["drinks_per_week"]
        if pd.isna(dpw):
            return np.nan
        if dpw == 0:
            return "none"
        elif 0 < dpw <= 7:
            return "low"
        elif 7 < dpw <= 14:
            return "moderate"
        else:
            return "heavy"

    alq_sub["alcohol_level"] = alq_sub.apply(categorize_alcohol, axis=1)

    alcohol_features = alq_sub[["id", "alcohol_level", "binge_drinker", "drinks_per_week"]].drop_duplicates("id")

    return alcohol_features


alcohol_features = build_alcohol_features(alq)
quick_overview(alcohol_features, "Features de alcohol")



=== Overview de Features de alcohol ===
Shape: (11268, 4)


,id,alcohol_level,binge_drinker,drinks_per_week
0,83732.0,low,NaN,1.00
1,83733.0,NaN,NaN,NaN
2,83734.0,NaN,NaN,NaN
3,83735.0,low,NaN,0.75
4,83736.0,low,NaN,0.25



Info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11268 entries, 0 to 11267
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               11268 non-null  float64
 1   alcohol_level    3150 non-null   object 
 2   binge_drinker    1273 non-null   float64
 3   drinks_per_week  3150 non-null   float64
dtypes: float64(3), object(1)
memory usage: 352.3+ KB
None


## 3.3 Sueño – SLQ

Objetivos:

1. Seleccionar variables:
   - Horas habituales de sueño.
   - Frecuencia de ronquidos.
   - Pausas respiratorias observadas durante el sueño.
   - Somnolencia diurna excesiva.

2. Derivar:
   - `sleep_duration_cat` (≤5 h, 6–7 h, 7–8 h, >8 h).
   - `probable_OSA` (0/1) combinando:
     - Ronquidos frecuentes.
     - Pausas respiratorias observadas.
     - Somnolencia diurna significativa.

Lógica de `probable_OSA` (simplificada):

- Se considera *probable apnea obstructiva del sueño (OSA)* si:
  - Ronca con frecuencia (por ejemplo “frecuentemente” o “casi siempre”), **y**
  - Alguna vez alguien le ha observado pausas respiratorias/gasping, **o**
  - Presenta somnolencia diurna excesiva (p. ej. se queda dormido involuntariamente varias veces a la semana).

Esta definición no pretende ser diagnóstica, sino capturar un fenotipo de alto riesgo a partir de preguntas de cuestionario.


In [44]:
# 3.3 Derivación de features de sueño (SLQ)

def build_sleep_features(slq_df):
    cols = [SLEEP_COLS["id"]]
    for key in ["hours_sleep", "snoring_freq", "apnea_obs", "daytime_sleepiness"]:
        col = SLEEP_COLS.get(key)
        if col in slq_df.columns:
            cols.append(col)

    slq_sub = slq_df[cols].copy().rename(columns={SLEEP_COLS["id"]: "id"})

    # Limpiar missing
    for key in ["hours_sleep", "snoring_freq", "apnea_obs", "daytime_sleepiness"]:
        col = SLEEP_COLS.get(key)
        if col in slq_sub.columns:
            slq_sub[col] = nhanes_missing_to_nan(slq_sub[col])

    # ---------- Categoría de duración de sueño ----------
    hours_col = SLEEP_COLS.get("hours_sleep")
    slq_sub["sleep_duration_cat"] = np.nan
    if hours_col in slq_sub.columns:
        h = slq_sub[hours_col]
        slq_sub["sleep_duration_cat"] = pd.cut(
            h,
            bins=[0, 5, 7, 8, 24],
            labels=["<=5h", "6-7h", "7-8h", ">8h"],
            right=True,
            include_lowest=True,
        )

    # ---------- Probable OSA ----------
    snore_col = SLEEP_COLS.get("snoring_freq")
    apnea_col = SLEEP_COLS.get("apnea_obs")
    sleepy_col = SLEEP_COLS.get("daytime_sleepiness")

    # Supuesto (ajusta según codebook):
    # 1=Never, 2=Rarely, 3=Sometimes, 4=Often, 5=Almost always
    def probable_osa(row):
        snore = row.get(snore_col, np.nan)
        apnea = row.get(apnea_col, np.nan)
        sleepy = row.get(sleepy_col, np.nan)

        # Si no contestó a nada del bloque, devolvemos NaN
        if pd.isna(snore) and pd.isna(apnea) and pd.isna(sleepy):
            return np.nan

        snore_high = pd.notna(snore) and snore >= 3
        apnea_any = pd.notna(apnea) and apnea >= 2
        sleepy_high = pd.notna(sleepy) and sleepy >= 3

        if snore_high and (apnea_any or sleepy_high):
            return 1
        return 0

    slq_sub["probable_OSA"] = slq_sub.apply(probable_osa, axis=1)

    sleep_features = slq_sub[["id", "sleep_duration_cat", "probable_OSA"]].drop_duplicates("id")

    return sleep_features


sleep_features = build_sleep_features(slq)
quick_overview(sleep_features, "Features de sueño")



=== Overview de Features de sueño ===
Shape: (12488, 3)


,id,sleep_duration_cat,probable_OSA
0,83732.0,6-7h,0.0
1,83733.0,7-8h,0.0
2,83734.0,NaN,0.0
3,83735.0,6-7h,0.0
4,83736.0,NaN,0.0



Info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12488 entries, 0 to 12487
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   id                  12488 non-null  float64 
 1   sleep_duration_cat  9049 non-null   category
 2   probable_OSA        12481 non-null  float64 
dtypes: category(1), float64(2)
memory usage: 207.6 KB
None


## 3.4 Depresión – DPQ (PHQ-9)

El módulo DPQ contiene los 9 ítems del **PHQ-9**, escala ampliamente validada para síntomas depresivos.

Pasos:

1. Seleccionar los 9 ítems DPQ010–DPQ090.
2. Calcular:
   - `depression_score`: suma de los 9 ítems (rango 0–27).
   - `depression_severity`: categoría estándar:
     - 0–4: none
     - 5–9: mild
     - 10–14: moderate
     - 15–19: moderately_severe
     - 20–27: severe

La depresión se asocia a mayor riesgo de hipertensión por múltiples vías (activación simpática crónica, peor adherencia a estilos de vida saludables y medicación, etc.), por lo que es razonable incluirla como covariable en un modelo de riesgo.


In [45]:
# 3.4 Derivación de features de depresión (PHQ-9)

def build_depression_features(dpq_df):
    cols = [DPQ_ITEMS["id"]] + [
        c for c in DPQ_ITEMS["items"] if c in dpq_df.columns
    ]
    dpq_sub = dpq_df[cols].copy().rename(columns={DPQ_ITEMS["id"]: "id"})

    # Limpiar missing en cada ítem
    for c in DPQ_ITEMS["items"]:
        if c in dpq_sub.columns:
            dpq_sub[c] = nhanes_missing_to_nan(dpq_sub[c])

    existing_items = [c for c in DPQ_ITEMS["items"] if c in dpq_sub.columns]
    dpq_sub["depression_score"] = dpq_sub[existing_items].sum(axis=1, min_count=1)

    def phq9_severity(score):
        if pd.isna(score):
            return np.nan
        if score <= 4:
            return "none"
        elif score <= 9:
            return "mild"
        elif score <= 14:
            return "moderate"
        elif score <= 19:
            return "moderately_severe"
        else:
            return "severe"

    dpq_sub["depression_severity"] = dpq_sub["depression_score"].apply(phq9_severity)

    depression_features = dpq_sub[["id", "depression_score", "depression_severity"]].drop_duplicates("id")

    return depression_features


depression_features = build_depression_features(dpq)
quick_overview(depression_features, "Features de depresión (PHQ-9)")



=== Overview de Features de depresión (PHQ-9) ===
Shape: (11268, 3)


,id,depression_score,depression_severity
0,83732.0,1.0,none
1,83733.0,2.0,none
2,83734.0,1.0,none
3,83735.0,13.0,moderate
4,83736.0,8.0,mild



Info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11268 entries, 0 to 11267
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   11268 non-null  float64
 1   depression_score     10257 non-null  float64
 2   depression_severity  10257 non-null  object 
dtypes: float64(2), object(1)
memory usage: 264.2+ KB
None


## 4. Tratamiento de valores faltantes y códigos especiales

En el código anterior, antes de derivar cada *feature* se aplica una función genérica:

```python
nhanes_missing_to_nan(series)


In [46]:
def na_summary(df, name):
    na_counts = df.isna().sum()
    na_pct = df.isna().mean() * 100
    summary = pd.DataFrame({"n_na": na_counts, "pct_na": na_pct})
    print(f"\n=== NA summary: {name} ===")
    display(summary)

na_summary(smoking_features, "smoking_features")
na_summary(alcohol_features, "alcohol_features")
na_summary(sleep_features, "sleep_features")
na_summary(depression_features, "depression_features")



=== NA summary: smoking_features ===


,n_na,pct_na
id,0,0.000000
current_smoker,8944,65.165756
former_smoker,8944,65.165756
never_smoker,8944,65.165756
pack_years_aprox,11846,86.309654
secondhand_smoke_home,597,4.349727
secondhand_smoke_inside,9826,71.591985
smoke_inside_freq_cat,13274,96.714026



=== NA summary: alcohol_features ===


,n_na,pct_na
id,0,0.000000
alcohol_level,8118,72.044728
binge_drinker,9995,88.702520
drinks_per_week,8118,72.044728



=== NA summary: sleep_features ===


,n_na,pct_na
id,0,0.000000
sleep_duration_cat,3439,27.538437
probable_OSA,7,0.056054



=== NA summary: depression_features ===


,n_na,pct_na
id,0,0.000000
depression_score,1011,8.972311
depression_severity,1011,8.972311


## 5. Filtro de población analítica común

Para mantener coherencia con el dataset `nhanes_supervised_dataset` y con la definición previa de `HTN_label`, se aplican los mismos criterios de inclusión:

1. Edad ≥ 18 años (`age_years >= 18`).
2. Participantes **examinados en el MEC** (`RIDSTATR == 2` en el dataset original DEMO; en el dataset `core` se asume que este filtro ya fue aplicado).

En este cuaderno:

- Partimos del dataset `core` (que ya debería estar filtrado a adultos examinados).
- Nos aseguramos de mantener únicamente las IDs presentes en `core` al integrar las nuevas features de cuestionario.


In [47]:
# Verificar shape inicial del core
print("Shape inicial de core:", core.shape)

# Filtro de adultos
if "age_years" in core.columns:
    core_filtered = core[core["age_years"] >= 18].copy()
else:
    core_filtered = core.copy()

print("Shape tras filtro de edad >= 18:", core_filtered.shape)

# Mapa de IDs válidos
valid_ids = set(core_filtered["id"].unique())
len(valid_ids)


Shape inicial de core: (11268, 34)
Shape tras filtro de edad >= 18: (11268, 34)


11268

## 6. Integración de módulos y construcción de `nhanes_features_rich`

En esta sección:

1. Se integran todas las features derivadas (tabaco, alcohol, sueño, depresión, comorbilidades) con el dataset base `core_filtered`.
2. Se documenta el tipo de *join* utilizado:
   - Se usa **left join** desde `core_filtered` hacia cada conjunto de features:
     - Esto garantiza que no se pierdan participantes del análisis principal, aunque falten respuestas en algún módulo.
3. Se revisan valores faltantes en el DataFrame final.
4. Se guardan **solo las columnas necesarias para el modelado** en:

   - `data/05_model_input/nhanes_features_rich.csv`.


In [48]:
# 6.1 Selección de columnas base demográficas/antropométricas

base_cols = [
    "id",
    "age_years",
    "gender_code",
    "race_ethnicity_code",
    "education_level_code",
    "income_poverty_ratio",
    "bmi",
    "waist_cm",
    "sedentary_minutes",
    "cycle",          # si existe
    "HTN_label",      # etiqueta de hipertensión (para modelado supervisado)
]

base_cols = [c for c in base_cols if c in core_filtered.columns]

features_rich = core_filtered[base_cols].copy()
print("Shape de features_rich (solo base):", features_rich.shape)


Shape de features_rich (solo base): (11268, 11)


In [49]:
# 6.2 Integración secuencial con las features derivadas (left joins)

features_rich = (
    features_rich
    .merge(smoking_features, on="id", how="left")
    .merge(alcohol_features, on="id", how="left")
    .merge(sleep_features, on="id", how="left")
    .merge(depression_features, on="id", how="left")
)

print("Shape de features_rich tras integrar módulos:", features_rich.shape)
quick_overview(features_rich, "features_rich (preview)")


Shape de features_rich tras integrar módulos: (11268, 25)

=== Overview de features_rich (preview) ===
Shape: (11268, 25)


,id,age_years,gender_code,race_ethnicity_code,education_level_code,income_poverty_ratio,bmi,waist_cm,sedentary_minutes,cycle,...,secondhand_smoke_home,secondhand_smoke_inside,smoke_inside_freq_cat,alcohol_level,binge_drinker,drinks_per_week,sleep_duration_cat,probable_OSA,depression_score,depression_severity
0,83732,62,1,3,5,4.39,27.8,101.1,480.0,2015_2016,...,0.0,NaN,NaN,low,NaN,1.00,6-7h,0.0,1.0,none
1,83733,53,1,3,3,1.32,30.8,107.9,300.0,2015_2016,...,1.0,1.0,NaN,NaN,NaN,NaN,7-8h,0.0,2.0,none
2,83734,78,1,3,3,1.51,28.8,116.5,480.0,2015_2016,...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,0.0,1.0,none
3,83735,56,2,3,5,5.00,42.4,110.1,480.0,2015_2016,...,0.0,NaN,NaN,low,NaN,0.75,6-7h,0.0,13.0,moderate
4,83736,42,2,4,4,1.23,20.3,80.4,540.0,2015_2016,...,1.0,1.0,1-3_days,low,NaN,0.25,NaN,0.0,8.0,mild



Info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11268 entries, 0 to 11267
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   id                       11268 non-null  int64   
 1   age_years                11268 non-null  int64   
 2   gender_code              11268 non-null  int64   
 3   race_ethnicity_code      11268 non-null  int64   
 4   education_level_code     11268 non-null  int64   
 5   income_poverty_ratio     11268 non-null  float64 
 6   bmi                      11268 non-null  float64 
 7   waist_cm                 11268 non-null  float64 
 8   sedentary_minutes        11268 non-null  float64 
 9   cycle                    11268 non-null  object  
 10  HTN_label                11268 non-null  int64   
 11  current_smoker           4551 non-null   float64 
 12  former_smoker            4551 non-null   float64 
 13  never_smoker             4551 non-null   float64 
 1

In [50]:
# 6.3 Revisión de NA en el dataset final

na_summary(features_rich, "features_rich (final)")


=== NA summary: features_rich (final) ===


,n_na,pct_na
id,0,0.000000
age_years,0,0.000000
gender_code,0,0.000000
race_ethnicity_code,0,0.000000
education_level_code,0,0.000000
income_poverty_ratio,0,0.000000
bmi,0,0.000000
waist_cm,0,0.000000
sedentary_minutes,0,0.000000
cycle,0,0.000000


### 1.1. Tabaco (`smoking_features`)

**current_smoker / former_smoker / never_smoker**  
- 0 % de valores faltantes.  
- Las tres variables están definidas para todos los sujetos, por lo que la clasificación básica de estado tabáquico está completa.

**pack_years_aprox**  
- ~86 % de valores faltantes.  
- Esto es esperable, porque solo puede calcularse en quienes:
  1. Han fumado de forma regular alguna vez.
  2. Tienen registrada la edad de inicio (SMD030).
  3. Y, si son exfumadores, disponen de edad de cese (SMD055) y/o cigarrillos/día (SMD650).  
- Dado ese conjunto de requisitos, es normal que solo una fracción de la muestra tenga un valor válido.

**secondhand_smoke_home / secondhand_smoke_inside**  
- 0 % de valores faltantes.  
- Con los códigos SMD460/SMD470 es sencillo mapear a dos categorías: “hay al menos una persona que fuma en el hogar” (≥1) versus “nadie fuma” (0).

**smoke_inside_freq_cat**  
- ~96 % de valores faltantes.  
- El problema está en el diseño del cuestionario, no en el código:
  - La pregunta SMD480 solo se formula si existe alguien que fuma dentro de la casa.
  - Para la mayoría (código 0 en SMD470: “nadie fuma dentro de la casa”), el ítem se omite y queda como missing.

---

### 1.2. Alcohol (`alcohol_features`)

**alcohol_level y binge_drinker**  
- 0 % de valores faltantes.

**drinks_per_week**  
- ~72 % de valores faltantes.  
- Es coherente con ALQ120Q/U: muchas personas no informan frecuencia o no aplican porque no bebieron en los últimos 12 meses.

**Advertencia importante**  
- En la implementación actual, muchos `NaN` de `drinks_per_week` se están traduciendo automáticamente a:
  - `alcohol_level = "none"`
  - `binge_drinker = 0`  
- En realidad, en estos casos **no se dispone de información suficiente** (“no sabemos”), por lo que esta recodificación introduce sesgo al asumir ausencia de consumo o de binge drinking.

---

### 1.3. Sueño (`sleep_features`)

**sleep_duration_cat**  
- ~28 % de valores faltantes.  
- Es un patrón habitual: una proporción de participantes no responde SLD010H (horas habituales de sueño).

**probable_OSA**  
- 0 % de valores faltantes.  
- Esto sugiere que, cuando las tres variables de base (ronquidos, pausas respiratorias, somnolencia diurna) están todas en `NaN`, la función está devolviendo 0 (no OSA) en lugar de un valor que refleje “desconocido”.  
- De nuevo, se trata de un problema de tratamiento de missing, porque se está interpretando falta de datos como ausencia de trastorno.

---

### 1.4. Depresión (`depression_features`)

**depression_score / depression_severity**  
- ~9 % de valores faltantes.  
- Es un nivel razonable y manejable de missing para las variables derivadas del PHQ-9.

---

### 1.5. Dataset final `features_rich`

Los patrones de valores faltantes del dataset final reflejan la combinación de todas las secciones anteriores. Destacan especialmente:

- `pack_years_aprox`: ~83 % de `NaN`.
- `smoke_inside_freq_cat`: ~95 % de `NaN`.
- `drinks_per_week`: ~72 % de `NaN`.
- `sleep_duration_cat`: ~28 % de `NaN`.

Estos porcentajes deben tenerse en cuenta al decidir qué variables incluir en los modelos y qué estrategias de imputación o categorización aplicar.


In [ ]:
# 6.4 Guardar dataset final

output_dir = "data/05_model_input"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "nhanes_features_rich.csv")
features_rich.to_csv(output_path, index=False)

print(f"Dataset 'nhanes_features_rich' guardado en: {output_path}")


## 7. Encapsulación en función para futura integración con Kedro

Para facilitar la integración en una *pipeline* de Kedro, encapsulamos la lógica principal en una función `build_features_rich(...)` que:

- Recibe:
  - `core_df`: DataFrame base con variables demográficas/antropométricas y `HTN_label`.
  - `smq_df`, `smqfam_df`, `alq_df`, `slq_df`, `dpq_df`, `mcq_df`: módulos de cuestionario.
- Devuelve:
  - `features_rich`: DataFrame final con todas las features integradas, listo para ser escrito a disco o a un *dataset* de Kedro.


In [ ]:
def build_features_rich(core_df, smq_df, smqfam_df, alq_df, slq_df, dpq_df, mcq_df):
    """
    Función envolvente para reproducir el flujo completo de este cuaderno.
    Pensada para ser usada como nodo en Kedro.
    """
    # Asegurar filtro de adultos
    if "age_years" in core_df.columns:
        core_f = core_df[core_df["age_years"] >= 18].copy()
    else:
        core_f = core_df.copy()

    # Reconstruir features por módulo
    smoking_f = build_smoking_features(smq_df, smqfam_df, core_f)
    alcohol_f = build_alcohol_features(alq_df)
    sleep_f = build_sleep_features(slq_df)
    depression_f = build_depression_features(dpq_df)

    # Columnas base
    base_cols_local = [
        "id",
        "age_years",
        "gender_code",
        "race_ethnicity_code",
        "education_level_code",
        "income_poverty_ratio",
        "bmi",
        "waist_cm",
        "sedentary_minutes",
        "cycle",
        "HTN_label",
    ]
    base_cols_local = [c for c in base_cols_local if c in core_f.columns]
    features = core_f[base_cols_local].copy()

    # Integración
    features = (
        features
        .merge(smoking_f, on="id", how="left")
        .merge(alcohol_f, on="id", how="left")
        .merge(sleep_f, on="id", how="left")
        .merge(depression_f, on="id", how="left")
    )

    return features


# Ejemplo de uso con los datos cargados en este cuaderno:
features_rich_check = build_features_rich(core, smq, smqfam, alq, slq, dpq)
print("Shape features_rich_check:", features_rich_check.shape)
